In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [4]:
import os

# Define the path to the Dataset folder
folder_path = '/content/drive/MyDrive/Sanskrit_LLM/Dataset'

# List contents of the folder to identify the file
print(f"Contents of {folder_path}:")
if os.path.exists(folder_path):
    for item in os.listdir(folder_path):
        print(item)
else:
    print(f"Folder not found: {folder_path}")

Contents of /content/drive/MyDrive/Sanskrit_LLM/Dataset:
DATASET.gsheet
disease_ontology.json
Raw_texts
Book_Names.json
NATIONAL AYURVEDA MORBIDITY CODES.xlsx
Sample_KG_Data
charaka_dataset.json
charaka_tree_dataset.json
charaka_tree_cleaned.json
Top Diseases.gsheet


In [5]:
# Install gspread if not already installed
!pip install gspread oauth2client

import gspread
from google.colab import auth
import pandas as pd
import google.auth # Import google.auth

# Authenticate Google Colab to access Google Drive and Sheets
# This will prompt you to authorize access in your browser.
auth.authenticate_user()

# Get default credentials and authorize gspread with them
creds, _ = google.auth.default()
gc = gspread.authorize(creds)

try:
    # Open the spreadsheet by its name. Assuming the actual Google Sheet is also named 'DATASET'.
    # If the sheet has a different name, you will need to update this.
    spreadsheet_name = 'DATASET'
    worksheet = gc.open(spreadsheet_name).worksheet("IAST_1000")

    # Get all values from the worksheet as a list of lists
    data = worksheet.get_all_values()

    # Convert to pandas DataFrame
    # The first row is assumed to be the header
    df_gsheet = pd.DataFrame(data[1:], columns=data[0])

    print(f"Successfully read Google Sheet '{spreadsheet_name}' into a DataFrame.")
    display(df_gsheet.head())

except gspread.exceptions.SpreadsheetNotFound:
    print(f"Error: Google Sheet '{spreadsheet_name}' not found. Please ensure the name is correct and you have access to it.")
except Exception as e:
    print(f"An error occurred while reading the Google Sheet: {e}")

Successfully read Google Sheet 'DATASET' into a DataFrame.


,IAST_term,English_meaning,input_text,target_text,ByT5_Output,Sanskrit5_Output,Srvm_op_test,Ind_ByT5_Combined_Hindi_OP,Ind_ByT5_Sanskrit_OP,Indic_Sanskrit_OP,Qwen_IAST_OP,Qwen_Devanagari_OP,Nalanda_Devanagari_OP
0,hikkā,hiccup,symptom: hiccup,hikkā,,hiccup_hiccup_SNM,हिक्का,हिचकी,,हिचकः ।,kṣudra-kṣumba,क्षुद्र-क्षुम्ब,"स्तत्स्: {ऽतोकेन्स्चोन्सुमेद्ऽ: ५१२, ऽतोकेन्स्..."
1,śvāsa,breathlessness/difficult breathing,symptom: breathlessness/difficult breathing,śvāsa,,breathlessness_breathlessness_U __different_U ...,श्वसनकष्टम्/कठिनश्वासः,सांस फूलना / सांस लेने में कठिनाई,symptom / Sanskrit Hindi,श्वासोच्छ्वासस्य अभावः / कष्टप्रदः श्वासोच्छ्वासः,kṣvabhāva,क्ष्वभाव,"स्तत्स्: {ऽतोकेन्स्चोन्सुमेद्ऽ: ५१२, ऽतोकेन्स्..."
2,cakṣurādīnāmupaghāta,impairment of eye,symptom: impairment of sense organs viz. eye,cakṣurādīnāmupaghāta,,impairment_ob_U eye_eye_U,नेत्रक्षतिः,आँखों की हानि,translation,नेत्रदोषः ।,śūdrāna,शूद्रान,"स्तत्स्: {ऽतोकेन्स्चोन्सुमेद्ऽ: ५१२, ऽतोकेन्स्..."
3,pīnasa,"cold, catarrh","symptom: cold, catarrh",pīnasa,,cold_cold_U,"शीत, श्लेष्म",शीत पित्त,translation,शीतकालः - शीतकालः - शीतकालः - शीतकालः - शीतकाल...,śirāśira,शिराशिर,"स्तत्स्: {ऽतोकेन्स्चोन्सुमेद्ऽ: ५१२, ऽतोकेन्स्..."
4,ardita,facial paralysis,symptom: facial paralysis,ardita,,facial_paralysis_,मुखपक्षाघातः,चेहरे का पक्षाघात,symptom,मुखस्य पक्षाघातः ।,ardita,अर्दित,"स्तत्स्: {ऽतोकेन्स्चोन्सुमेद्ऽ: ५१२, ऽतोकेन्स्..."


In [6]:
# -----------------------------
# Install aksharamukha
# -----------------------------
!pip install aksharamukha

# -----------------------------
# LOAD MODEL (NLLB)
# -----------------------------
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from aksharamukha import transliterate
import torch
from tqdm import tqdm

model_name = "facebook/nllb-200-distilled-600M"

tokenizer = AutoTokenizer.from_pretrained(model_name, src_lang="eng_Latn")
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 289.9/289.9 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.1/183.1 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.9/531.9 kB 22.3 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [7]:
def translate_batch_nllb(text_list):

    inputs = [
        f"Ayurvedic symptom description: {str(t)}"
        for t in text_list
    ]

    tokenized = tokenizer(
        inputs,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128
    ).to(device)

    outputs = model.generate(
        **tokenized,
        forced_bos_token_id=tokenizer.convert_tokens_to_ids("san_Deva"),
        max_length=100,
        num_beams=4
    )

    devanagari = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    iast = [
        transliterate.process("Devanagari", "IAST", d)
        for d in devanagari
    ]

    return devanagari, iast



In [8]:
# -----------------------------
# APPLY ON DATAFRAME
# -----------------------------
batch_size = 8   # NLLB is heavy → keep small

texts = df_gsheet["English_meaning"].fillna("").astype(str).tolist()

dev_results = []
iast_results = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i+batch_size]

    dev_batch, iast_batch = translate_batch_nllb(batch)

    dev_results.extend(dev_batch)
    iast_results.extend(iast_batch)


# -----------------------------
# STORE OUTPUT
# -----------------------------
df_gsheet["NLLB_Devanagari_OP"] = dev_results
df_gsheet["NLLB_IAST_OP"] = iast_results

df_gsheet.head()

100%|██████████| 125/125 [03:01<00:00,  1.45s/it]


,IAST_term,English_meaning,input_text,target_text,ByT5_Output,Sanskrit5_Output,Srvm_op_test,Ind_ByT5_Combined_Hindi_OP,Ind_ByT5_Sanskrit_OP,Indic_Sanskrit_OP,Qwen_IAST_OP,Qwen_Devanagari_OP,Nalanda_Devanagari_OP,NLLB_Devanagari_OP,NLLB_IAST_OP
0,hikkā,hiccup,symptom: hiccup,hikkā,,hiccup_hiccup_SNM,हिक्का,हिचकी,,हिचकः ।,kṣudra-kṣumba,क्षुद्र-क्षुम्ब,"स्तत्स्: {ऽतोकेन्स्चोन्सुमेद्ऽ: ५१२, ऽतोकेन्स्...",आयुर्वेदिकं लक्षणवर्णनं - हिकुकम् ।,āyurvedikaṃ lakṣaṇavarṇanaṃ - hikukam .
1,śvāsa,breathlessness/difficult breathing,symptom: breathlessness/difficult breathing,śvāsa,,breathlessness_breathlessness_U __different_U ...,श्वसनकष्टम्/कठिनश्वासः,सांस फूलना / सांस लेने में कठिनाई,symptom / Sanskrit Hindi,श्वासोच्छ्वासस्य अभावः / कष्टप्रदः श्वासोच्छ्वासः,kṣvabhāva,क्ष्वभाव,"स्तत्स्: {ऽतोकेन्स्चोन्सुमेद्ऽ: ५१२, ऽतोकेन्स्...",आयुर्वेदिकं लक्षणवर्णनं- श्वासं न श्वासः/ श्वा...,āyurvedikaṃ lakṣaṇavarṇanaṃ- śvāsaṃ na śvāsaḥ/...
2,cakṣurādīnāmupaghāta,impairment of eye,symptom: impairment of sense organs viz. eye,cakṣurādīnāmupaghāta,,impairment_ob_U eye_eye_U,नेत्रक्षतिः,आँखों की हानि,translation,नेत्रदोषः ।,śūdrāna,शूद्रान,"स्तत्स्: {ऽतोकेन्स्चोन्सुमेद्ऽ: ५१२, ऽतोकेन्स्...",आयुर्वेदिकं लक्षणवर्णनं- नेत्रदोषः ।,āyurvedikaṃ lakṣaṇavarṇanaṃ- netradoṣaḥ .
3,pīnasa,"cold, catarrh","symptom: cold, catarrh",pīnasa,,cold_cold_U,"शीत, श्लेष्म",शीत पित्त,translation,शीतकालः - शीतकालः - शीतकालः - शीतकालः - शीतकाल...,śirāśira,शिराशिर,"स्तत्स्: {ऽतोकेन्स्चोन्सुमेद्ऽ: ५१२, ऽतोकेन्स्...","आयुर्वेदिकं लक्षणवर्णनं - सर्दी, कैटार्ह","āyurvedikaṃ lakṣaṇavarṇanaṃ - sardī, kaiṭārha"
4,ardita,facial paralysis,symptom: facial paralysis,ardita,,facial_paralysis_,मुखपक्षाघातः,चेहरे का पक्षाघात,symptom,मुखस्य पक्षाघातः ।,ardita,अर्दित,"स्तत्स्: {ऽतोकेन्स्चोन्सुमेद्ऽ: ५१२, ऽतोकेन्स्...",आयुर्वेदिकं लक्षणवर्णनं- अनुहारस्य पक्षाघातः ।,āyurvedikaṃ lakṣaṇavarṇanaṃ- anuhārasya pakṣāg...


In [10]:
spreadsheet = gc.open("DATASET")
worksheet = spreadsheet.worksheet("IAST_1000")

worksheet.clear()

data_to_upload = [df_gsheet.columns.values.tolist()] + df_gsheet.values.tolist()
worksheet.update(data_to_upload)

{'spreadsheetId': '17Z102gEEQakxuEc2t4qLLmXDrFld8Vn-AyHKkv4Vbfo',
 'updatedRange': 'IAST_1000!A1:O1000',
 'updatedRows': 1000,
 'updatedColumns': 15,
 'updatedCells': 15000}